In [17]:
import torch
import random
import torch.nn.functional as F
import matplotlib.pyplot as plt  # for making figures

%matplotlib inline

# Boilerplate code from lecture
Ref: https://colab.research.google.com/drive/1WV2oi2fh9XXyldh02wupFQX0wh5ZC-z-?usp=sharing#scrollTo=MJPU8HT08PPu

In [18]:
# read in all the words
words = open("names.txt", "r").read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [19]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [20]:
# build the dataset
block_size = (
    3  # context length: how many characters do we take to predict the next one?
)


def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]  # crop and append

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y


random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])  # 80%
Xdev, Ydev = build_dataset(words[n1:n2])  # 10%
Xte, Yte = build_dataset(words[n2:])  # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


# Writing your own gradients

In [21]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(
        f"{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}"
    )

In [165]:
# Initialize parameters of the network
# taken from our previous notebook: makermore_ml_activations_gradients.ipynb

n_emb = 10
n_hidden = 64
context_length = 3
vocab_size = len(chars) + 1

g = torch.Generator().manual_seed(6)
C = torch.randn((vocab_size, n_emb), generator=g)
W1 = (
    torch.randn((n_emb * context_length, n_hidden), generator=g)
    * (5 / 3)
    / ((n_emb * context_length) ** 0.5)
)
b1 = torch.randn(n_hidden, generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b2 = torch.randn(vocab_size, generator=g) * 0

# batchnorm parameter initialization
# we want to start with gain = 1 and bias = 0. These params will be learned while training
# in PyTorch nomenclature, bngain (gamma) and bnbias (beta) are referred to as "buffers"
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))

# # running mean and std for running predictions post-training [no grad required]
# bnmean_running = torch.zeros((1, n_hidden))
# bnstd_running = torch.ones((1, n_hidden))


parameters = [C, W1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True

In [166]:
# Start with one minibatch
batch_size = 32
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]  # batch X,Y

##### Batchnorm broken down into parts in code below
\[
\begin{aligned}
\mu_{\text{batch}} &= \frac{1}{m}\sum_{i=1}^{m} \text{activation}_i \\
\text{diff}i &= \text{activation}i - \mu_{\text{batch}} \\
\sigma^2_{\text{batch}} &= \frac{1}{m-1}\sum_{i=1}^{m} \text{diff}_i^{\,2} \\
\text{activation\_norm}i &= \frac{\text{diff}i}{\sqrt{\sigma^2_{\text{batch}} + \epsilon}}, \quad \epsilon = 10^{-5}
\end{aligned}
\]

##### Loss function broken down into parts
\[
\begin{aligned}
\text{max\_logit}i &= \max{j} \, \text{logits}{ij} \\
\text{centered\_logit}{ij} &= \text{logits}{ij} - \text{max\_logit}i \\
\text{exp\_logit}{ij} &= \exp(\text{centered\_logit}{ij}) \\
\text{total\_exp}i &= \sum{j} \text{exp\_logit}{ij} \\
\text{prob}{ij} &= \frac{\text{exp\_logit}{ij}}{\text{total\_exp}i} \\
\text{log\_prob}{ij} &= \log(\text{prob}{ij}) \\
\mathcal{L} &= -\frac{1}{m} \sum_{i=1}^{m} \text{log\_prob}_{i, \, y_i}
\end{aligned}
\]

In PyTorch, for a given example, the max logit is subtracted from every logit in its row for numerical stability. Overall formula is equivalent to:

\[
\begin{aligned}
\mathcal{L} = -\frac{1}{m}\sum_{i=1}^{m} \log\left(\frac{e^{\text{logits}_{i, y_i} - \text{max\_logit}i}}{\sum_j e^{\text{logits}{ij} - \text{max\_logit}_i}}\right)
\end{aligned}
\]

In [167]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time
# ref: https://colab.research.google.com/drive/1WV2oi2fh9XXyldh02wupFQX0wh5ZC-z-

emb = C[Xb]  # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1)  # concatenate the vectors

# Linear layer 1
hprebn = embcat @ W1 + b1  # hidden layer pre-activation

# BatchNorm layer
bnmeani = 1 / batch_size * hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = (
    1 / (batch_size - 1) * (bndiff2).sum(0, keepdim=True)
)  # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias

# Non-linearity
h = torch.tanh(hpreact)  # hidden layer

# Linear layer 2
logits = h @ W2 + b2  # output layer

# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes  # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = (
    counts_sum**-1
)  # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[
    range(batch_size), Yb
].mean()  # for each sample, pick the log-prob assigned to the correct label

# PyTorch backward pass
for p in parameters:
    p.grad = None
for t in [
    logprobs,
    probs,
    counts,
    counts_sum,
    counts_sum_inv,  # afaik there is no cleaner way
    norm_logits,
    logit_maxes,
    logits,
    h,
    hpreact,
    bnraw,
    bnvar_inv,
    bnvar,
    bndiff2,
    bndiff,
    hprebn,
    bnmeani,
    embcat,
    emb,
]:
    t.retain_grad()
loss.backward()
loss

tensor(3.2913, grad_fn=<NegBackward0>)

In [259]:
# Exercise 1: backprop through the whole thing manually,
# backpropagating through exactly all of the variables
# as they are defined in the forward pass above, one by one

# -----------------

# (1) shape of a variable and its gradient will be the same. dlogprobs.shape will have to = logprobs.shape
# (2) next, in logprobs we pick specific neuron's output for every example
# so only those (example, neuron) pairs will have a gradient, rest will be 0
# (3) loss = -(a + b + c)/3 --> dloss/da = -1/3
# forward: loss = -logprobs[range(batch_size), Yb].mean()
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(batch_size), Yb] = -1.0 / batch_size  # because mean()
cmp("logprobs", dlogprobs, logprobs)

# forward: logprobs = probs.log()
dprobs = dlogprobs * 1.0 / probs
cmp("probs", dprobs, probs)

# forward: probs = counts * counts_sum_inv
# probs is (32, 27), counts is (32, 27), counts_sum_inv is (32, 1)
# counts_sum_inv represents inv(sum total of all neurons for each example)
# IMPLICIT broadcasting: pytorch takes counts_sum_inv and replicates it 27 times -- once per neuron, to compute probs.
# dcounts_sum_inv.shape should also be (32, 1)
# hence we sum the gradients of neurons for each example to obtain one gradient per example
dcounts_sum_inv = (dprobs * counts).sum(1, keepdim=True)
cmp("counts_sum_inv", dcounts_sum_inv, counts_sum_inv)

# forward: probs = counts * counts_sum_inv
# probs is (32, 27), counts is (32, 27), counts_sum_inv is (32, 1)
# we know dcounts.shape should be (32, 27)
# NOTE: cannot check dcounts right now since "counts" node is used twice:
# (1) prods = counts * counts_sum_inv and (2) counts_sum = counts.sum(1, keepdims=True)
dcounts = counts_sum_inv * dprobs

# forward: counts_sum_inv = counts_sum**-1
# counts_sum_inv and counts_sum are both (32, 1)
dcounts_sum = dcounts_sum_inv * (-(counts_sum**-2))
cmp("counts_sum", dcounts_sum, counts_sum)

# forward: counts_sum = counts.sum(1, keepdims=True)
# counts_sum is (32, 1) and counts in (32, 27)
# example: y = (x11+x21+x31)
# dl/dx11 = dl/dy * d(x11 + x21 + x31)/dx11 --> dl/dx11 = dl/dy * 1
dcounts += dcounts_sum * torch.ones_like(counts)
cmp("counts", dcounts, counts)

# forward: counts = norm_logits.exp()
# counts and norm_logits are both (32, 27)
# given y=e^x --> dl/dx = dl/dy * dy/dx = dl/dy * e^x = dl/dy * y
dnorm_logits = dcounts * counts
cmp("norm_logits", dnorm_logits, norm_logits)

# forward: norm_logits = logits - logit_maxes
# norm_logits is (32, 27), logits is (32, 27), logit_maxes = (32, 1)
dlogit_maxes = -1.0 * dnorm_logits.sum(1, keepdim=True)
cmp("logit_maxes", dlogit_maxes, logit_maxes)

# forward: norm_logits = logits - logit_maxes
# norm_logits is (32, 27), logits is (32, 27), logit_maxes is (32, 1)
# NOTE: used twice: (1) norm_logits = logits - logit_maxes, (2) logit_maxes = logits.max(1, keepdim=True).values
dlogits = dnorm_logits.clone()

# forward: logit_maxes = logits.max(1, keepdim=True).values
# logit_maxes is (32, 1) and logits is (32, 27)
# NOTE: for a given example, we should only update gradient for the neuron that had max logit
dlogits_intermediate = torch.zeros_like(logits)
dlogits_intermediate[range(batch_size), logits.max(1).indices] = 1
# dlogits_intermediate = F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) # Karpathy's code
# plt.imshow(dlogits_intermediate)
dlogits += dlogits_intermediate * dlogit_maxes
cmp("logits", dlogits, logits)

# forward: logits = h @ W2 + b2
# logits is (32, 27), h is (32, 64), W2 is (64, 27), b2 is (27)
# calculating an example on paper will reveal a clean pattern
dh = dlogits @ W2.T
cmp("h", dh, h)

dW2 = h.T @ dlogits
cmp("W2", dW2, W2)

db2 = dlogits.sum(0, keepdim=False)
cmp("b2", db2, b2)

# forward: h = torch.tanh(hpreact)
# dh and hpreact are both (32, 64)
dhpreact = dh * (1 - h**2)
cmp("hpreact", dhpreact, hpreact)

# forward: hpreact = bngain * bnraw + bnbias
# dhpreact (32, 64), bngain (1, 64), bnraw (32, 64), bnbias (1, 64)
dbngain = (dhpreact * bnraw).sum(0, keepdim=True)
cmp("bngain", dbngain, bngain)

dbnbias = dhpreact.sum(0, keepdim=True)
cmp("bnbias", dbnbias, bnbias)

dbnraw = dhpreact * bngain
cmp("bnraw", dbnraw, bnraw)

# forward: bnraw = bndiff * bnvar_inv
# dnraw (32, 64), bndiff (32, 64), bnvar_inv (1, 64)
# NOTE: bndiff shows up twice: (1) bnraw = bndiff * bnvar_inv and (2) bndiff2 = bndiff**2

dbnvar_inv = (dbnraw * bndiff).sum(0, keepdim=True)
cmp("bnvar_inv", dbnvar_inv, bnvar_inv)

dbndiff = dbnraw * bnvar_inv

# forward: bnvar_inv = (bnvar + 1e-5) ** -0.5
# dbnvar_inv and bnvar both (1, 64)
dbnvar = dbnvar_inv * (-0.5 * (bnvar + 1e-5) ** -1.5)
cmp("bnvar", dbnvar, bnvar)

# forward: bnvar = 1 / (batch_size - 1) * (bndiff2).sum(0, keepdim=True)
# dbnvar (1, 64), bndiff2 (32, 64)
dbndiff2 = dbnvar * (1 / (batch_size - 1) * torch.ones_like(bndiff2))
cmp("bndiff2", dbndiff2, bndiff2)

# forward: bndiff2 = bndiff**2
# bndiff2 and bndiff (32, 64)
dbndiff += dbndiff2 * 2 * bndiff
cmp("bndiff", dbndiff, bndiff)

# forward: bndiff = hprebn - bnmeani
# bndiff (32, 64), hprebn (32, 64), bnmeani (1, 64)
# NOTE hprebn shows up twice: (1) bndiff = hprebn - bnmeani, (2) bnmeani = 1 / batch_size * hprebn.sum(0, keepdim=True)
dbnmeani = -dbndiff.sum(0, keepdim=True)
cmp("bnmeani", dbnmeani, bnmeani)

dhprebn = dbndiff.clone()

# forward: bnmeani = 1 / batch_size * hprebn.sum(0, keepdim=True)
# bnmeani (1, 64), hprebn (32, 64)
dhprebn += dbnmeani * (1 / batch_size * torch.ones_like(hprebn))
cmp("hprebn", dhprebn, hprebn)

# forward: hprebn = embcat @ W1 + b1
# dhprebn (32, 64), embcat (32, 30), W1 (30, 64), b1 (64)

dembcat = dhprebn @ W1.T
cmp("embcat", dembcat, embcat)

dW1 = embcat.T @ dhprebn
cmp("W1", dW1, W1)

db1 = dhprebn.sum(0, keepdim=False)
# cmp("b1", db1, b1)

# forward: embcat = emb.view(emb.shape[0], -1)
# embcat (32, 30), emb (32, 3, 10)
demb = dembcat.view(emb.shape)
cmp("emb", demb, emb)

# forward: emb = C[Xb]
# emb (32, 3, 10), C (27, 10), Xb (32, 3)
# dC = torch.zeros_like(C)
# dC[set(Xb.flatten().tolist()), :] = 1

# -----------------

# cmp('logprobs', dlogprobs, logprobs)
# cmp('probs', dprobs, probs)
# cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
# cmp('counts_sum', dcounts_sum, counts_sum)
# cmp('counts', dcounts, counts)
# cmp('norm_logits', dnorm_logits, norm_logits)
# cmp('logit_maxes', dlogit_maxes, logit_maxes)
# cmp('logits', dlogits, logits)
# cmp('h', dh, h)
# cmp('W2', dW2, W2)
# cmp('b2', db2, b2)
# cmp('hpreact', dhpreact, hpreact)
# cmp('bngain', dbngain, bngain)
# cmp('bnbias', dbnbias, bnbias)
# cmp('bnraw', dbnraw, bnraw)
# cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
# cmp('bnvar', dbnvar, bnvar)
# cmp('bndiff2', dbndiff2, bndiff2)
# cmp('bndiff', dbndiff, bndiff)
# cmp('bnmeani', dbnmeani, bnmeani)
# cmp('hprebn', dhprebn, hprebn)
# cmp('embcat', dembcat, embcat)
# cmp('W1', dW1, W1)
# cmp('b1', db1, b1)
# cmp('emb', demb, emb)
# cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff:

In [273]:
# forward: emb = C[Xb]
# emb (32, 3, 10), C (27, 10), Xb (32, 3)

dC = torch.zeros_like(C)
Xb_flat = Xb.flatten().tolist()
demb_flat = demb.view(96, 10)

# for char_idx in Xb_flat:
for i, char_idx in enumerate(Xb_flat):
    # dC[char_idx, :] += demb_flat[char_idx, :]
    dC[char_idx] += demb_flat[i]

cmp("C", dC, C)

C               | exact: True  | approximate: True  | maxdiff: 0.0


In [266]:
dC = torch.zeros_like(C)
dC.index_add_(0, Xb.flatten(), demb.view(-1, 10))
cmp("C", dC, C)

C               | exact: True  | approximate: True  | maxdiff: 0.0
